# Station Stacking v7 - KDAL

Current experimental notebook for `KDAL`.

This version trains on live-safe same-day 11 AM GFS/HRRR timing, direct 13Z NBM raw-high data, source-owned v6 trend feature inputs, expanding year folds, and durable Optuna SQLite storage. Artifacts are written to `data/calibration/station_stacking_v7`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KDAL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 50
STACK_OPTUNA_TRIALS = 50
OPTUNA_STARTUP_TRIALS = 20
STACK_OPTUNA_STARTUP_TRIALS = 20
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.calibration.station_stacking import (
    StationStackingConfig,
    V7_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V7 Contract

`feature_version="v7"` applies the source-owned v5 feature block and keeps the 11 AM observation trend columns. `timing_mode="same_day_11am_live_safe"` selects forecast cycles that would have been available by the bot decision time, while the current-observation trend cache falls back to the existing 11 AM observation timing.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]

V7_FEATURE_COLUMNS


['v2_recent_heat_anomaly_f',
 'v2_recent_heat_momentum_f',
 'v2_morning_warmup_to_consensus_f',
 'v2_consensus_minus_7d_actual_f',
 'v2_spread_per_warmup_f',
 'v2_humidity_warmup_interaction',
 'v3_high_so_far_above_current_f',
 'v3_remaining_warmup_from_high_so_far_f',
 'v3_high_so_far_minus_lag_1d_f',
 'v3_high_so_far_minus_7d_actual_f',
 'v3_remaining_warmup_per_spread_f',
 'v3_humidity_remaining_warmup_interaction',
 'v4_forecast_precip_total_mean_mm',
 'v4_forecast_precip_total_max_mm',
 'v4_forecast_precip_total_spread_mm',
 'v4_forecast_precip_max_1h_mean_mm',
 'v4_forecast_precip_hours_mean',
 'v4_forecast_precip_intensity_mean',
 'v4_forecast_precip_intensity_max',
 'v4_any_forecast_precip',
 'v4_all_forecast_precip',
 'v4_observed_precip_any',
 'v4_observed_precip_recent_mm_est',
 'v4_forecast_total_minus_observed_recent_mm',
 'v4_forecast_observed_precip_match',
 'v4_forecast_wet_observed_dry',
 'v4_observed_wet_forecast_dry',
 'v4_precip_humidity_interaction',
 'v4_precip_r

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
6,KDAL,gfs,1983,2021-01-01,2026-06-10
7,KDAL,hrrr,1987,2021-01-01,2026-06-10
8,KDAL,nbm,1986,2021-01-01,2026-06-10


## Model Scores


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v7",
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v7/KDAL_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-14 10:09:22,911] A new study created in RDB with name: KDAL_v7_base_xgboost_mae_f
[I 2026-06-14 10:09:32,004] Trial 0 finished with value: 1.6509606094313845 and parameters: {'n_estimators': 1342, 'learning_rate': 0.19043899115982607, 'max_depth': 9, 'min_child_weight': 2.481040974867813, 'gamma': 2.340279606636548, 'subsample': 0.4513964382185317, 'colsample_bytree': 0.3877543479093296, 'reg_alpha': 2.478071022662141, 'reg_lambda': 0.6132587025321562}. Best is trial 0 with value: 1.6509606094313845.
[I 2026-06-14 10:11:27,353] Trial 1 finished with value: 1.7734345972914158 and parameters: {'n_estimators': 2493, 'learning_rate': 0.001120367191095075, 'max_depth': 12, 'min_child_weight': 21.368329072358772, 'gamma': 3.185086660174142, 'subsample': 0.4681862286846154, 'colsample_bytree': 0.46921293140473197, 'reg_alpha': 4.476173538513514e-07, 'reg_lambda': 0.20253776634919213}. Best is trial 0 with value: 1.6509606094313845.
[I 2026-06-14 10:12:57,176] Trial 2 finished with 

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,668,1.453471,2.097286
1,validation_2024_2025,lightgbm,668,1.541176,2.147034
2,validation_2024_2025,catboost,668,1.512469,2.158059
3,validation_2024_2025,hrrr_raw,668,5.486824,6.274688
4,validation_2024_2025,gfs_raw,668,2.718801,3.724769
5,test_2026,xgboost,128,1.556840,2.037201
6,test_2026,lightgbm,128,1.559188,2.007716
7,test_2026,catboost,128,1.503384,1.938182
8,test_2026,ridge_stack,128,1.508933,1.963582
9,test_2026,hrrr_raw,128,5.859180,6.536264


## NBM Raw High


In [8]:
nbm_raw_metrics = result.metrics.loc[result.metrics["method"].eq("nbm_raw")].copy()
nbm_raw_metrics


,evaluation_scope,method,count,mae_f,rmse_f,bias_f,within_1f_pct,within_2f_pct,within_3f_pct,first_contract_date,last_contract_date
4,year_split_test,nbm_raw,128,2.157190,2.831328,1.388977,33.593750,57.03125,75.000000,2026-01-01,2026-05-18
10,year_split_validation,nbm_raw,668,2.671242,3.546488,2.170693,20.209581,44.91018,66.916168,2024-01-01,2025-12-31


## Morning Trend Coverage


In [9]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,100.0
1,observed_temp_change_last_3h_f,100.0
2,observed_morning_warmup_rate_f_per_hour,100.0
3,observed_high_so_far_change_since_9am_f,100.0


In [10]:
result.feature_columns.loc[result.feature_columns["feature"].isin(TREND_COLUMNS)]


,feature,kind
23,observed_temp_change_last_1h_f,numeric
24,observed_temp_change_last_3h_f,numeric
25,observed_morning_warmup_rate_f_per_hour,numeric
26,observed_high_so_far_change_since_9am_f,numeric


## Rounded Within 1F Accuracy


In [11]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
0,oof_2026,catboost,128,77,60.156250
5,oof_2026,ridge_stack,128,75,58.593750
6,oof_2026,xgboost,128,75,58.593750
3,oof_2026,lightgbm,128,71,55.468750
4,oof_2026,nbm_raw,128,59,46.093750
1,oof_2026,gfs_raw,128,38,29.687500
2,oof_2026,hrrr_raw,128,11,8.593750
12,validation_2024_2025,xgboost,668,427,63.922156
7,validation_2024_2025,catboost,668,415,62.125749
10,validation_2024_2025,lightgbm,668,402,60.179641


## Version Comparison


In [12]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,catboost,128,1.503384,1.938182,v7
1,test_2026,ridge_stack,128,1.508933,1.963582,v7
2,test_2026,xgboost,128,1.556840,2.037201,v7
3,test_2026,lightgbm,128,1.559188,2.007716,v7
4,test_2026,ridge_stack,127,1.574416,2.107321,v6
...,...,...,...,...,...,...
61,validation_2024_2025,hrrr_raw,660,5.477361,6.266007,v3
62,validation_2024_2025,hrrr_raw,668,5.486824,6.274688,v7
63,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v5
64,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v6


## 2026 OOF Weather Brackets


In [13]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,128,1.556840,2.037201,43.75
1,lightgbm,128,1.559188,2.007716,36.71875
2,catboost,128,1.503384,1.938182,39.84375
3,ridge_stack,128,1.508933,1.963582,43.75
4,hrrr_raw,128,5.859180,6.536264,5.46875
5,gfs_raw,128,2.848730,3.526536,19.53125
